### References

*   [https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876](https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876)
*   [https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo](https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo)
*   [https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/](https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/)
*   https://www.kaggle.com/code/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch
*   [https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference](https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference)
*   https://www.kaggle.com/code/neibyr/30-min-just-use-semantic-search-qwen3-emb-0-6b
*   https://www.kaggle.com/code/datafan07/jigsaw-speed-run-10-min-triplet-and-faiss
*   https://www.kaggle.com/code/nahidhossainredom/deberta-v3-base-3-epochs-lb-0-906

In [ ]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

In [ ]:
# !uv pip install transformers==4.45.2

In [ ]:
%%writefile constants.py
BASE_MODEL_PATH = "/kaggle/input/qwen2.5/transformers/0.5b-instruct-gptq-int4/1"
LORA_PATH = "output/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

In [ ]:
%%writefile utils.py
import pandas as pd
import re

def url_to_semantics(text: str) -> str:
    if not isinstance(text, str):
        return ""

    url_pattern = r'https?://[^\s/$.?#].[^\s]*'
    urls = re.findall(url_pattern, text)

    if not urls:
        return ""

    all_semantics = []
    seen_semantics = set()

    for url in urls:
        url_lower = url.lower()

        domain_match = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", url_lower)
        if domain_match:
            full_domain = domain_match.group(1)
            parts = full_domain.split('.')
            for part in parts:
                if part and part not in seen_semantics and len(part) > 3: # Avoid short parts like 'www'
                    all_semantics.append(f"domain:{part}")
                    seen_semantics.add(part)

        # 2. Extract path parts
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", url_lower)
        path_parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()] # Split by common delimiters

        for part in path_parts:
            # Clean up potential file extensions or query params
            part_clean = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", part)
            if part_clean and part_clean not in seen_semantics and len(part_clean) > 3:
                all_semantics.append(f"path:{part_clean}")
                seen_semantics.add(part_clean)

    if not all_semantics:
        return ""

    return f"\nURL Keywords: {' '.join(all_semantics)}"


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []

    flatten.append(train_dataset[["body", "rule", "subreddit","rule_violation"]].copy())

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"

            if col_name in train_dataset.columns:
                sub_dataset = train_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0

                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]

                if not sub_dataset.empty:
                    flatten.append(sub_dataset)

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"

            if col_name in test_dataset.columns:
                sub_dataset = test_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0

                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]

                if not sub_dataset.empty:
                    flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=['body', 'rule', 'subreddit'], ignore_index=True)
    dataframe.drop_duplicates(subset=['body','rule'],keep='first',inplace=True)

    return dataframe.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel
from transformers.modeling_outputs import SequenceClassifierOutput

class JigsawModelTextW(nn.Module):
    def __init__(self, model_name: str, num_labels: int,  num_freeze_layer: int = 12):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size * 2, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

        if num_freeze_layer > 0:
            self._freeze_top_half_layers(num_freeze_layer)

    def _freeze_top_half_layers(self, num_freeze_layer):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = num_freeze_layer

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def _extract_text_tokens(self, token_type_ids, attention_mask, last_hidden_state):
        """token_type_idsを使用してtext部分を抽出"""
        batch_size = token_type_ids.size(0)
        text_embeddings = []

        for i in range(batch_size):
            # text部分（token_type_id == 1）のマスクを作成
            text_mask = (token_type_ids[i] == 1) & (attention_mask[i] == 1)

            if text_mask.any():
                # text部分の埋め込みを抽出
                text_tokens = last_hidden_state[i][text_mask]
                text_pooled = text_tokens.mean(dim=0)
            else:
                text_pooled = torch.zeros(self.config.hidden_size, device=token_type_ids.device)

            text_embeddings.append(text_pooled)

        return torch.stack(text_embeddings)


    def _pool_text_only(self, token_type_ids, attention_mask, last_hidden_state):
        # last_hidden_state: [B,S,H], attention_mask: [B,S], token_type_ids: [B,S] or None
        use_type_ids = (
            token_type_ids is not None
            and token_type_ids.dim() == 2
            and torch.any(token_type_ids > 0)
        )
        if use_type_ids:
            mask_2d = (token_type_ids == 1) & (attention_mask == 1)  # [B,S] boolean AND
        else:
            raise Exception

        mask = mask_2d.unsqueeze(-1).type_as(last_hidden_state)      # [B,S,1] -> broadcast OK
        num = (last_hidden_state * mask).sum(dim=1)                  # [B,H]
        den = mask.sum(dim=1).clamp(min=1e-9)                        # [B,1]
        return num / den

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        sequence_output, _ = outputs.last_hidden_state.max(1)  # tuple: [layer0..last]

        # Text部分のみのpooling
        text_features = self._pool_text_only(
            token_type_ids, attention_mask, outputs.last_hidden_state
        )  # [batch_size, hidden_size]

        # concat
        sequence_output = torch.cat([sequence_output, text_features], dim=1)
        logits = self.regressor(sequence_output)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)


class JigsawModelConcatrate(nn.Module):
    def __init__(self, model_name: str, num_labels: int, num_freeze_layer = 12):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size * 4, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

        if num_freeze_layer > 0:
            self._freeze_top_half_layers()

    def _freeze_top_half_layers(self,  num_freeze_layer):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = num_freeze_layer

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        sequence_output = torch.cat([outputs["hidden_states"][-1*i][:,0] for i in range(1, 4+1)], dim=1)  # concatenate
        logits = self.regressor(sequence_output)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

class JigsawModelCustomHeader(nn.Module):
    def __init__(self, model_name: str, header:str, num_labels: int,  num_freeze_layer: int = 12):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size, num_labels)
        self.header = header

        if self.header == "lstm":
            self.lstm = nn.LSTM(self.config.hidden_size, self.config.hidden_size, batch_first=True)
        elif self.header == "attention":
            # Attention layer
            self.attention = nn.MultiheadAttention(
                embed_dim=self.config.hidden_size,
                num_heads=8,  # 通常8または16
                dropout=0.1,
                batch_first=True  # 重要: batch_first=True
            )

            # Layer normalization and dropout
            self.layer_norm = nn.LayerNorm(self.config.hidden_size)
            self.dropout = nn.Dropout(0.1)

        # 共通
        self.regressor = nn.Linear(self.config.hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()


        if  num_freeze_layer > 0:
            self._freeze_top_half_layers(num_freeze_layer)

    def _freeze_top_half_layers(self, num_freeze_layer: int = 12):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = num_freeze_layer

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        if self.header == "maxpooling":
            sequence_output, _ = outputs['last_hidden_state'].max(1)  # max pooling
        elif self.header == "meanpooling":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs['last_hidden_state'].size()).float()
            sequence_output = torch.sum(outputs['last_hidden_state'] * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        elif self.header == "lstm":
            out, _ = self.lstm(outputs['last_hidden_state'], None)
            sequence_output = out[:, -1, :]
        elif self.header == "attention":
            # 最後の隠れ状態を取得
            last_hidden_state = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

            # Self-attention適用
            # key_padding_maskでPADトークンをマスク
            key_padding_mask = ~attention_mask.bool()  # PADトークンをTrue

            attn_output, attn_weights = self.attention(
                query=last_hidden_state,
                key=last_hidden_state,
                value=last_hidden_state,
                key_padding_mask=key_padding_mask
            )

            # Residual connection + Layer Norm
            attn_output = self.layer_norm(attn_output + last_hidden_state)
            attn_output = self.dropout(attn_output)

            # [CLS]トークン（最初のトークン）を取得
            sequence_output = attn_output[:, 0, :]  # [batch_size, hidden_size]

        logits = self.regressor(sequence_output)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)


In [ ]:
%%writefile model_configs.py
from dataclasses import dataclass
from typing import Optional, Dict, Any, List
import os

@dataclass
class ModelConfig:
    """統一モデル設定クラス"""
    name: str
    model_type: str  # "deberta", "qwen", "triplet"
    enabled: bool = True
    weight: float = 1.0

    # DeBERTa specific
    model_path: Optional[str] = None
    max_length: int = 256
    batch_size: int = 4
    epochs: int = 3
    learning_rate: float = 2e-5
    gradient_accumulation_steps: int = 4
    use_irm: bool  =True
    irm_lambda: float = 100.0
    irm_n_envs: int = 3,
    model_class: Optional[str] = None
    model_kwargs: Optional[Dict[str, Any]] = None

    # Qwen specific
    base_model_path: Optional[str] = None
    lora_path: str = "qwen_output/"
    positive_answer: str = "Yes"
    negative_answer: str = "No"

    # Triplet specific
    embedding_model_path: Optional[str] = None
    triplet_epochs: int = 1
    triplet_batch_size: int = 16
    margin: float = 0.25

    # Common paths
    data_path: str = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir: Optional[str] = None

    def __post_init__(self):
        if self.output_dir is None:
            self.output_dir = f"./model_{self.name}"
        if self.model_kwargs is None:
            self.model_kwargs = {}

class ModelRegistry:
    """モデル設定のレジストリ"""

    @staticmethod
    def get_default_configs() -> List[ModelConfig]:
        return [
            # DeBERTa models
            ModelConfig(
                name="deberta_offensive_default",
                model_type="deberta",
                model_class="JigsawModelCustomHeader",
                model_kwargs={"header": "meanpooling", "num_freeze_layer": 0},
                model_path="/kaggle/input/huggingfacedebertav3variants/deberta-v3-large-offensive",
                max_length=256,
                batch_size=4,
                epochs=3,
                gradient_accumulation_steps=4,
                weight=0.20,
                use_irm=False,
                enabled=True
            ),
            ModelConfig(
                name="deberta_offensive_maxpool",
                model_type="deberta",
                model_class="JigsawModelCustomHeader",
                model_kwargs={"header": "maxpooling", "num_freeze_layer": 7},
                model_path="/kaggle/input/huggingfacedebertav3variants/deberta-v3-large-offensive",
                max_length=256,
                batch_size=4,
                epochs=3,
                gradient_accumulation_steps=4,
                weight=0.28,
                use_irm=True,
                irm_lambda=100.0,
                irm_n_envs=3,
                enabled=True
            ),
            ModelConfig(
                name="deberta_large_off_text",
                model_type="deberta",
                model_class="JigsawModelTextW",
                model_kwargs={"num_freeze_layer": 7},
                model_path="/kaggle/input/huggingfacedebertav3variants/deberta-v3-large-offensive",
                max_length=256,
                batch_size=4,
                epochs=3,
                gradient_accumulation_steps=4,
                weight=0.28,
                use_irm=True,
                irm_lambda=100.0,
                irm_n_envs=3,
                enabled=True
            ),

            # Qwen models
            ModelConfig(
                name="qwen_0_5b",
                model_type="qwen",
                base_model_path="/kaggle/input/qwen2.5/transformers/0.5b-instruct-gptq-int4/1",
                lora_path="qwen_output/",
                batch_size=4,
                epochs=1,
                weight=0.22,
                enabled=False  
            ),

            # Triplet model
            ModelConfig(
                name="triplet_bge",
                model_type="triplet",
                embedding_model_path="/kaggle/input/baai/transformers/bge-large-en-v1.5/1",
                triplet_epochs=1,
                triplet_batch_size=16,
                margin=0.25,
                weight=0.22,
                enabled=False
            ),
        ]

    @staticmethod
    def filter_enabled(configs: List[ModelConfig]) -> List[ModelConfig]:
        return [c for c in configs if c.enabled]

In [ ]:
%%writefile utils_training.py
import os
import pandas as pd
import torch
import torch.nn.functional as F
import random
import numpy as np
from transformers import AutoTokenizer, Trainer, TrainingArguments
from typing import Dict, Any, Optional, Tuple
from utils import get_dataframe_to_train, url_to_semantics

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def prepare_training_data(data_path: str) -> pd.DataFrame:
    """訓練データの準備（EDA拡張含む）"""

    training_df = get_dataframe_to_train(data_path)

    # URL semantic処理
    training_df['body_with_url'] = training_df['body'].apply(lambda x: x + url_to_semantics(x))

    return training_df

def prepare_tokenized_data(
    df: pd.DataFrame,
    tokenizer: AutoTokenizer,
    max_length: int,
) -> Dict[str, Any]:
    """データのトークン化"""
    # rule と body を別々に渡す
    rules = df['rule'].tolist()
    bodies = df['body_with_url'].tolist()
    encodings = tokenizer(
        rules, bodies,
        truncation=True, padding=True, max_length=max_length,
        return_token_type_ids=True
    )

    if "environment_id" in df.columns:
        encodings["environment_id"] = df["environment_id"].astype(int).tolist()

    return encodings

class IRMTrainer(Trainer):
    def __init__(self, *args, irm_lambda: float = 100.0, irm_n_envs: int = 2, **kwargs):
        super().__init__(*args, **kwargs)
        self.irm_lambda = irm_lambda
        self.irm_n_envs = irm_n_envs

    def compute_loss(self, model, inputs, num_items_in_batch=None, return_outputs=False):
        # print(inputs)
        labels = inputs.pop("labels")
        env_ids = inputs.pop("environment_id")  # [B]
        outputs = model(**inputs)
        logits = outputs.logits  # [B, C]

        # 環境ごとにERMとΩを計算
        losses = []
        penalties = []
        for e in range(self.irm_n_envs):
            mask = (env_ids == e)
            if mask.any():
                l = F.cross_entropy(logits[mask], labels[mask])
                losses.append(l)

                # IRMv1: ダミースカラー s による勾配ペナルティ
                s = torch.tensor(1.0, device=logits.device, requires_grad=True)
                l_s = F.cross_entropy(s * logits[mask], labels[mask])
                g = torch.autograd.grad(l_s, s, create_graph=True)[0]
                penalties.append(g.pow(2))
        if len(losses) == 0:
            loss_erm = F.cross_entropy(logits, labels)
            penalty = logits.new_tensor(0.0)
        else:
            loss_erm = torch.stack(losses).mean()
            penalty = torch.stack(penalties).mean()

        loss = loss_erm + self.irm_lambda * penalty
        return (loss, outputs) if return_outputs else loss

def create_trainer(
    model: torch.nn.Module,
    train_dataset: torch.utils.data.Dataset,
    output_dir: str,
    epochs: int = 3,
    learning_rate: float = 2e-5,
    batch_size: int = 8,
    gradient_accumulation_steps: int = 1,
    use_irm: bool = False,
    irm_lambda: float = 100.0,
    irm_n_envs: int = 2,
    **kwargs
) -> Trainer:
    """Trainerインスタンスの作成"""
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_ratio=0.1,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",
        logging_steps=10,
        fp16=True,
        remove_unused_columns=False,
        # max_steps=30, # debug
        **kwargs
    )
    if use_irm:
        trainer = IRMTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            irm_lambda=irm_lambda,
            irm_n_envs=irm_n_envs,
        )
    else:
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
        )

    return trainer

def predict_and_save(
    trainer: Trainer,
    test_df: pd.DataFrame,
    tokenizer: AutoTokenizer,
    max_length: int,
    output_filename: str
) -> None:
    """テストデータの予測と保存"""
    from utils import url_to_semantics

    # テストデータの準備
    test_df = test_df.copy()
    test_df['body_with_url'] = test_df['body'].apply(lambda x: x + url_to_semantics(x))

    # トークン化
    test_encodings = prepare_tokenized_data(
        test_df, tokenizer, max_length
    )
    test_dataset = JigsawDataset(test_encodings)

    # 予測
    predictions = trainer.predict(test_dataset)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

    # 保存
    submission_df = pd.DataFrame({
        "row_id": test_df["row_id"],
        "rule_violation": probs
    })
    submission_df.to_csv(output_filename, index=False)
    print(f"✓ Predictions saved to {output_filename}")

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None, env_ids=None):
        self.encodings = encodings
        self.labels = labels
        self.env_ids = env_ids

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        
        if self.env_ids:
            item["environment_id"] = torch.tensor(self.env_ids[idx])
        if self.labels:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

In [ ]:
%%writefile qwen_trainer.py
import os
import pandas as pd
import random
import numpy as np
import multiprocessing as mp
from typing import Dict, Any
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from transformers.utils import is_torch_bf16_gpu_available
import vllm
import torch
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest

class QwenModelTrainer:
    def __init__(self, config):
        self.config = config
        random.seed(42)
        np.random.seed(42)

    def build_prompt(self, row):
        """プロンプトテンプレート構築"""
        base_prompt = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''
        return f"""
{base_prompt}

Subreddit: r/{row["subreddit"]}
Rule: {row["rule"]}
Examples:
1) {row["positive_example"]}
Answer: Yes

2) {row["negative_example"]}
Answer: No

---
Comment: {row["body"]}
Answer:"""

    def get_dataframe_to_train(self):
        """学習用データフレーム準備"""
        train_dataset = pd.read_csv(f"{self.config.data_path}/train.csv")
        test_dataset = pd.read_csv(f"{self.config.data_path}/test.csv").sample(frac=0.5, random_state=42).reset_index(drop=True)

        flatten = []

        # 训练集处理
        train_df = train_dataset[["body", "rule", "subreddit", "rule_violation",
                                  "positive_example_1","positive_example_2",
                                  "negative_example_1","negative_example_2"]].copy()

        train_df["positive_example"] = np.where(
            np.random.rand(len(train_df)) < 0.5,
            train_df["positive_example_1"],
            train_df["positive_example_2"]
        )
        train_df["negative_example"] = np.where(
            np.random.rand(len(train_df)) < 0.5,
            train_df["negative_example_1"],
            train_df["negative_example_2"]
        )
        train_df.drop(columns=["positive_example_1","positive_example_2",
                               "negative_example_1","negative_example_2"], inplace=True)
        flatten.append(train_df)

        # 测试集处理
        for violation_type in ["positive", "negative"]:
            for i in range(1, 3):
                sub_dataset = test_dataset[["rule","subreddit",
                                            "positive_example_1","positive_example_2",
                                            "negative_example_1","negative_example_2"]].copy()

                if violation_type == "positive":
                    body_col = f"positive_example_{i}"
                    other_positive_col = f"positive_example_{3-i}"
                    sub_dataset["body"] = sub_dataset[body_col]
                    sub_dataset["positive_example"] = sub_dataset[other_positive_col]
                    sub_dataset["negative_example"] = np.where(
                        np.random.rand(len(sub_dataset)) < 0.5,
                        sub_dataset["negative_example_1"],
                        sub_dataset["negative_example_2"]
                    )
                    sub_dataset["rule_violation"] = 1
                else:
                    body_col = f"negative_example_{i}"
                    other_negative_col = f"negative_example_{3-i}"
                    sub_dataset["body"] = sub_dataset[body_col]
                    sub_dataset["negative_example"] = sub_dataset[other_negative_col]
                    sub_dataset["positive_example"] = np.where(
                        np.random.rand(len(sub_dataset)) < 0.5,
                        sub_dataset["positive_example_1"],
                        sub_dataset["positive_example_2"]
                    )
                    sub_dataset["rule_violation"] = 0

                sub_dataset.drop(columns=["positive_example_1","positive_example_2",
                                          "negative_example_1","negative_example_2"], inplace=True)
                flatten.append(sub_dataset)

        dataframe = pd.concat(flatten, axis=0)
        return dataframe.drop_duplicates(ignore_index=True)

    def build_dataset(self, dataframe):
        """学習用データセット構築"""
        dataframe["prompt"] = dataframe.apply(self.build_prompt, axis=1)
        columns = ["prompt"]
        if "rule_violation" in dataframe:
            dataframe["completion"] = dataframe["rule_violation"].map({
                1: self.config.positive_answer,
                0: self.config.negative_answer,
            })
            columns.append("completion")

        dataframe = dataframe[columns]
        dataset = Dataset.from_pandas(dataframe)
        return dataset

    def train(self):
        """Qwenモデル学習実行"""
        print(f"🚀 Training Qwen model: {self.config.name}")

        dataframe = self.get_dataframe_to_train()
        train_dataset = self.build_dataset(dataframe)

        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            task_type="CAUSAL_LM",
        )

        training_args = SFTConfig(
            num_train_epochs=self.config.epochs,
            per_device_train_batch_size=self.config.batch_size,
            gradient_accumulation_steps=4,
            optim="paged_adamw_8bit",
            learning_rate=1e-4,
            weight_decay=0.01,
            max_grad_norm=1.0,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            bf16=is_torch_bf16_gpu_available(),
            fp16=not is_torch_bf16_gpu_available(),
            dataloader_pin_memory=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            save_strategy="no",
            report_to="none",
            completion_only_loss=True,
            packing=False,
            remove_unused_columns=False,
            # max_steps=100 # debug
        )

        trainer = SFTTrainer(
            self.config.base_model_path,
            args=training_args,
            train_dataset=train_dataset,
            peft_config=lora_config,
        )

        trainer.train()
        trainer.save_model(self.config.lora_path)
        print(f"✅ Qwen model saved to {self.config.lora_path}")

    def worker_inference(self, device_id, df_slice, return_dict):
        """推論ワーカー（マルチプロセス用）"""
        os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
        os.environ["VLLM_USE_V1"] = "0"
        os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

        llm = vllm.LLM(
            self.config.base_model_path,
            quantization="gptq",
            tensor_parallel_size=1,
            gpu_memory_utilization=0.98,
            trust_remote_code=True,
            dtype="half",
            enforce_eager=True,
            max_model_len=2836,
            disable_log_stats=True,
            enable_prefix_caching=True,
            enable_lora=True,
            max_lora_rank=64,
        )

        tokenizer = llm.get_tokenizer()
        mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[self.config.positive_answer, self.config.negative_answer])

        test_dataset = self.build_dataset(df_slice)
        texts = test_dataset["prompt"]

        outputs = llm.generate(
            texts,
            vllm.SamplingParams(
                skip_special_tokens=True,
                max_tokens=1,
                logits_processors=[mclp],
                logprobs=2,
            ),
            use_tqdm=True,
            lora_request=LoRARequest("default", 1, self.config.lora_path)
        )

        log_probs = [
            {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
            for out in outputs
        ]
        predictions = pd.DataFrame(log_probs)[[self.config.positive_answer, self.config.negative_answer]]
        predictions["row_id"] = df_slice["row_id"].values
        return_dict[device_id] = predictions

    def predict(self, output_filename: str):
        """Qwenモデル推論実行"""
        print(f"🔮 Making Qwen predictions: {self.config.name}")

        test_dataframe = pd.read_csv(f"{self.config.data_path}/test.csv")

        # 随机选择例子
        test_dataframe["positive_example"] = test_dataframe.apply(
            lambda row: random.choice([row["positive_example_1"], row["positive_example_2"]]),
            axis=1
        )
        test_dataframe["negative_example"] = test_dataframe.apply(
            lambda row: random.choice([row["negative_example_1"], row["negative_example_2"]]),
            axis=1
        )
        test_dataframe = test_dataframe.drop(
            columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"],
            errors="ignore"
        )

        # データセット作成
        test_dataset = self.build_dataset(test_dataframe)
        texts = test_dataset["prompt"]

        # tp_size = max(1, torch.cuda.device_count())
        # print(f"Using tensor_parallel_size={tp_size}")

        llm = vllm.LLM(
            self.config.base_model_path,
            quantization="gptq",
            tensor_parallel_size=1,
            gpu_memory_utilization=0.98,
            trust_remote_code=True,
            dtype="half",
            enforce_eager=True,
            max_model_len=2836,
            disable_log_stats=True,
            enable_prefix_caching=True,
            enable_lora=True,
            max_lora_rank=64,
        )

        tokenizer = llm.get_tokenizer()
        mclp = MultipleChoiceLogitsProcessor(
            tokenizer, choices=[self.config.positive_answer, self.config.negative_answer]
        )

        # まとめて生成（必要なら分割）
        outputs = llm.generate(
            texts,
            vllm.SamplingParams(
                skip_special_tokens=True,
                max_tokens=1,
                logits_processors=[mclp],
                logprobs=2,
            ),
            use_tqdm=True,
            lora_request=LoRARequest("default", 1, self.config.lora_path),
        )

        log_probs = [
            {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
            for out in outputs
        ]
        predictions = pd.DataFrame(log_probs)[[self.config.positive_answer, self.config.negative_answer]]
        predictions["row_id"] = test_dataframe["row_id"].values

        # 提出ファイル作成（rank正規化）
        submission = predictions[["row_id", self.config.positive_answer]].rename(
            columns={self.config.positive_answer: "rule_violation"}
        )
        rq = submission["rule_violation"].rank(method="average") / (len(submission) + 1)
        submission["rule_violation"] = rq

        submission.to_csv(output_filename, index=False)
        print(f"✅ Qwen predictions saved to {output_filename}")

In [ ]:
%%writefile triplet_trainer.py
import os
import pandas as pd
import numpy as np
import random
import re
from urllib.parse import urlparse
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer, SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments, models
)
from sentence_transformers.losses import TripletLoss
from sklearn.cluster import AgglomerativeClustering
from umap import UMAP
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

class TripletModelTrainer:
    def __init__(self, config):
        self.config = config
        random.seed(42)
        np.random.seed(42)

    def cleaner(self, text):
        """Replace URLs with format: <url>: (domain/important-path)"""
        if not text:
            return text
        url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'
        def replace_url(match):
            url = match.group(0)
            try:
                parsed = urlparse(url)
                domain = parsed.netloc.lower()
                if domain.startswith('www.'):
                    domain = domain[4:]
                path_parts = [part for part in parsed.path.split('/') if part]
                if path_parts:
                    important_path = '/'.join(path_parts[:2])
                    return f"<url>: ({domain}/{important_path})"
                else:
                    return f"<url>: ({domain})"
            except:
                return "<url>: (unknown)"
        return re.sub(url_pattern, replace_url, str(text))

    def load_test_data(self):
        print("Loading test data...")
        test_df = pd.read_csv(f'{self.config.data_path}/test.csv')
        print(f"Loaded {len(test_df)} test examples")
        return test_df

    def collect_all_texts(self, test_df):
        print("\nCollecting all texts for embedding...")
        all_texts = set()
        for body in test_df['body']:
            if pd.notna(body):
                all_texts.add(self.cleaner(str(body)))
        example_cols = ['positive_example_1', 'positive_example_2',
                        'negative_example_1', 'negative_example_2']
        for col in example_cols:
            for example in test_df[col]:
                if pd.notna(example):
                    all_texts.add(self.cleaner(str(example)))
        all_texts = list(all_texts)
        print(f"Collected {len(all_texts)} unique texts")
        return all_texts

    def create_test_triplet_dataset(self, test_df, augmentation_factor=2, subsample_fraction=1.0):
        anchors, positives, negatives = [], [], []
        print("Creating rule-aligned triplets from test data...")

        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing test rows"):
            rule = self.cleaner(str(row['rule']))
            pos_examples = []
            neg_examples = []

            for neg_col in ['negative_example_1', 'negative_example_2']:
                if pd.notna(row[neg_col]):
                    pos_examples.append(self.cleaner(str(row[neg_col])))
            for pos_col in ['positive_example_1', 'positive_example_2']:
                if pd.notna(row[pos_col]):
                    neg_examples.append(self.cleaner(str(row[pos_col])))

            for pos_ex in pos_examples:
                for neg_ex in neg_examples:
                    anchors.append(rule)
                    positives.append(pos_ex)
                    negatives.append(neg_ex)

        # Augmentation logic
        if augmentation_factor > 0:
            print(f"Adding {augmentation_factor}x augmentation...")
            rule_positives = {}
            rule_negatives = {}

            for rule in test_df['rule'].unique():
                rule_df = test_df[test_df['rule'] == rule]
                pos_pool, neg_pool = [], []

                for _, row in rule_df.iterrows():
                    for neg_col in ['negative_example_1', 'negative_example_2']:
                        if pd.notna(row[neg_col]):
                            pos_pool.append(self.cleaner(str(row[neg_col])))
                    for pos_col in ['positive_example_1', 'positive_example_2']:
                        if pd.notna(row[pos_col]):
                            neg_pool.append(self.cleaner(str(row[pos_col])))

                rule_positives[rule] = list(set(pos_pool))
                rule_negatives[rule] = list(set(neg_pool))

            for rule in test_df['rule'].unique():
                clean_rule = self.cleaner(str(rule))
                pos_pool = rule_positives[rule]
                neg_pool = rule_negatives[rule]
                n_samples = min(augmentation_factor * len(pos_pool), len(pos_pool) * len(neg_pool))

                for _ in range(n_samples):
                    if pos_pool and neg_pool:
                        anchors.append(clean_rule)
                        positives.append(random.choice(pos_pool))
                        negatives.append(random.choice(neg_pool))

        combined = list(zip(anchors, positives, negatives))
        random.shuffle(combined)

        if subsample_fraction < 1.0:
            n_samples = int(len(combined) * subsample_fraction)
            combined = combined[:n_samples]

        anchors, positives, negatives = zip(*combined) if combined else ([], [], [])
        print(f"Created {len(anchors)} triplets from test data")

        dataset = Dataset.from_dict({
            'anchor': list(anchors),
            'positive': list(positives),
            'negative': list(negatives)
        })
        return dataset

    def fine_tune_model(self, model, train_dataset):
        print(f"Fine-tuning model on {len(train_dataset)} triplets...")

        loss = TripletLoss(model=model, triplet_margin=self.config.margin)
        dataset_size = len(train_dataset)
        steps_per_epoch = max(1, dataset_size // self.config.triplet_batch_size)
        max_steps = steps_per_epoch * self.config.triplet_epochs

        output_dir = f"./models/test-finetuned-bge"
        args = SentenceTransformerTrainingArguments(
            output_dir=output_dir,
            num_train_epochs=self.config.triplet_epochs,
            per_device_train_batch_size=self.config.triplet_batch_size,
            warmup_steps=0,
            learning_rate=2e-5,
            logging_steps=max(1, max_steps // 4),
            save_strategy="epoch",
            save_total_limit=1,
            fp16=True,
            max_grad_norm=1.0,
            dataloader_drop_last=False,
            gradient_checkpointing=True,
            gradient_accumulation_steps=1,
            max_steps=max_steps,
            report_to="none"
        )

        trainer = SentenceTransformerTrainer(
            model=model, args=args, train_dataset=train_dataset, loss=loss
        )
        trainer.train()

        final_model_path = f"{output_dir}/final"
        print(f"Saving fine-tuned model to {final_model_path}...")
        model.save_pretrained(final_model_path)
        return model, final_model_path

    def load_or_create_finetuned_model(self, test_df):
        fine_tuned_path = "./models/test-finetuned-bge/final"

        if os.path.exists(fine_tuned_path):
            print(f"Loading existing fine-tuned model from {fine_tuned_path}...")
            model = SentenceTransformer(fine_tuned_path)
            model.half()
            return model

        print("Fine-tuned model not found. Creating new one...")

        try:
            word_embedding_model = models.Transformer(
                self.config.embedding_model_path, max_seq_length=256, do_lower_case=True
            )
            pooling_model = models.Pooling(
                word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean"
            )
            base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
            print("Loaded base model from Kaggle path with explicit pooling")
        except:
            # Fallback to HuggingFace
            base_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
            print("Loaded base model from HuggingFace")

        triplet_dataset = self.create_test_triplet_dataset(test_df, augmentation_factor=2, subsample_fraction=1.)
        fine_tuned_model, model_path = self.fine_tune_model(base_model, triplet_dataset)

        print(f"Fine-tuning completed. Model saved to: {model_path}")
        fine_tuned_model.half()
        return fine_tuned_model

    def generate_embeddings(self, texts, model, batch_size=64):
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = model.encode(
            sentences=texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_tensor=False,
            normalize_embeddings=True
        )
        return embeddings

    def create_rule_centroids_with_hierarchical_clustering(self, test_df, text_to_embedding, rule_embeddings):
        print(f"\nCreating rule centroids with Hierarchical Clustering + UMAP...")
        base_umap_components = 32
        rule_centroids = {}

        for rule in test_df['rule'].unique():
            rule_data = test_df[test_df['rule'] == rule]

            pos_embeddings = []
            for _, row in rule_data.iterrows():
                for col in ['positive_example_1', 'positive_example_2']:
                    if pd.notna(row[col]):
                        clean_text = self.cleaner(str(row[col]))
                        if clean_text in text_to_embedding:
                            pos_embeddings.append(text_to_embedding[clean_text])

            neg_embeddings = []
            for _, row in rule_data.iterrows():
                for col in ['negative_example_1', 'negative_example_2']:
                    if pd.notna(row[col]):
                        clean_text = self.cleaner(str(row[col]))
                        if clean_text in text_to_embedding:
                            neg_embeddings.append(text_to_embedding[clean_text])

            if pos_embeddings and neg_embeddings:
                pos_embeddings = np.array(pos_embeddings)
                neg_embeddings = np.array(neg_embeddings)

                # UMAP processing with crash prevention
                def maybe_umap(X):
                    n = X.shape[0]
                    if n > 10 and n > base_umap_components:
                        n_components_safe = min(base_umap_components, max(2, n - 2))
                        reducer = UMAP(n_components=n_components_safe, random_state=42)
                        return reducer.fit_transform(X)
                    else:
                        return X

                pos_reduced = maybe_umap(pos_embeddings)
                neg_reduced = maybe_umap(neg_embeddings)

                # Clustering logic
                n_pos_clusters = min(3, len(pos_embeddings))
                n_neg_clusters = min(3, len(neg_embeddings))

                pos_centroids = []
                neg_centroids = []

                if n_pos_clusters > 1:
                    pos_clusterer = AgglomerativeClustering(n_clusters=n_pos_clusters)
                    pos_labels = pos_clusterer.fit_predict(pos_reduced)
                    for cluster_id in np.unique(pos_labels):
                        cluster_mask = pos_labels == cluster_id
                        cluster_embeddings = pos_embeddings[cluster_mask]
                        cluster_centroid = cluster_embeddings.mean(axis=0)
                        cluster_centroid = cluster_centroid / np.linalg.norm(cluster_centroid)
                        pos_centroids.append(cluster_centroid)
                else:
                    pos_centroid = pos_embeddings.mean(axis=0)
                    pos_centroid = pos_centroid / np.linalg.norm(pos_centroid)
                    pos_centroids.append(pos_centroid)

                if n_neg_clusters > 1:
                    neg_clusterer = AgglomerativeClustering(n_clusters=n_neg_clusters)
                    neg_labels = neg_clusterer.fit_predict(neg_reduced)
                    for cluster_id in np.unique(neg_labels):
                        cluster_mask = neg_labels == cluster_id
                        cluster_embeddings = neg_embeddings[cluster_mask]
                        cluster_centroid = cluster_embeddings.mean(axis=0)
                        cluster_centroid = cluster_centroid / np.linalg.norm(cluster_centroid)
                        neg_centroids.append(cluster_centroid)
                else:
                    neg_centroid = neg_embeddings.mean(axis=0)
                    neg_centroid = neg_centroid / np.linalg.norm(neg_centroid)
                    neg_centroids.append(neg_centroid)

                rule_centroids[rule] = {
                    'positive_centroids': pos_centroids,
                    'negative_centroids': neg_centroids,
                    'pos_count': len(pos_embeddings),
                    'neg_count': len(neg_embeddings),
                    'rule_embedding': rule_embeddings[rule]
                }

        return rule_centroids

    def train(self):
        print(f"🚀 Training Triplet model: {self.config.name}")
        # Training is done in load_or_create_finetuned_model
        pass

    def predict(self, output_filename: str):
        print(f"🔮 Making Triplet predictions: {self.config.name}")

        test_df = self.load_test_data()
        model = self.load_or_create_finetuned_model(test_df)

        all_texts = self.collect_all_texts(test_df)
        all_embeddings = self.generate_embeddings(all_texts, model)
        text_to_embedding = {text: emb for text, emb in zip(all_texts, all_embeddings)}

        # Generate rule embeddings
        unique_rules = test_df['rule'].unique()
        rule_embeddings = {}
        for rule in unique_rules:
            clean_rule = self.cleaner(str(rule))
            rule_emb = model.encode(clean_rule, convert_to_tensor=False, normalize_embeddings=True)
            rule_embeddings[rule] = rule_emb

        rule_centroids = self.create_rule_centroids_with_hierarchical_clustering(
            test_df, text_to_embedding, rule_embeddings
        )

        # Prediction logic
        row_ids = []
        predictions = []
        for rule in test_df['rule'].unique():
            rule_data = test_df[test_df['rule'] == rule]
            if rule not in rule_centroids:
                continue

            pos_centroids = rule_centroids[rule]['positive_centroids']
            neg_centroids = rule_centroids[rule]['negative_centroids']

            for _, row in rule_data.iterrows():
                body = self.cleaner(str(row['body']))
                row_id = row['row_id']
                if body not in text_to_embedding:
                    continue

                body_embedding = text_to_embedding[body]

                pos_distances = [np.linalg.norm(body_embedding - pos_centroid)
                               for pos_centroid in pos_centroids]
                neg_distances = [np.linalg.norm(body_embedding - neg_centroid)
                               for neg_centroid in neg_centroids]

                min_pos_distance = min(pos_distances) if pos_distances else 1.0
                min_neg_distance = min(neg_distances) if neg_distances else 1.0
                rule_prediction = min_neg_distance - min_pos_distance

                row_ids.append(row_id)
                predictions.append(rule_prediction)

        submission_df = pd.DataFrame({
            'row_id': row_ids,
            'rule_violation': predictions
        })
        submission_df.to_csv(output_filename, index=False)
        print(f"✅ Triplet predictions saved to {output_filename}")

In [ ]:
import numpy as np
from utils_training import *
from models import *
from model_configs import ModelConfig, ModelRegistry
from qwen_trainer import QwenModelTrainer
from triplet_trainer import TripletModelTrainer
import warnings
warnings.filterwarnings('ignore')

class UnifiedModelTrainer:
    """統一モデル管理クラス"""

    def __init__(self, model_configs: list):
        self.model_configs = model_configs
        self.data_path = "/kaggle/input/jigsaw-agile-community-rules/"

    def train_single_deberta_model(self, config: ModelConfig) -> str:
        """DeBERTaモデルの訓練"""
        print(f"\n🚀 Training DeBERTa model: {config.name}")
        print("-" * 50)

        # データ準備
        training_df = prepare_training_data(self.data_path)
        test_df = pd.read_csv(f"{self.data_path}/test.csv")

        # IRM: subredditをK分割してenvironment_idを作成（簡易ハッシュ）
        if getattr(config, "use_irm", False):
            K = getattr(config, "irm_n_envs", 3)
            col = getattr(config, "irm_env_col", "subreddit")
            training_df["environment_id"] = training_df[col].astype(str).apply(lambda s: (hash(s) % K))

        # print(training_df.head(10))
        # トークナイザー
        tokenizer = AutoTokenizer.from_pretrained(config.model_path)

        # データセット準備
        train_encodings = prepare_tokenized_data(training_df, tokenizer, config.max_length)
        train_dataset = JigsawDataset(train_encodings, training_df['rule_violation'].tolist(), training_df["environment_id"].tolist() if getattr(config, "use_irm", False) else None)

        # モデル
        if config.model_class == "JigsawModelTextW":
            num_freeze_layer = config.model_kwargs.get("num_freeze_layer", 12)
            model = JigsawModelTextW(config.model_path, num_labels=2, num_freeze_layer=num_freeze_layer)
        elif config.model_class == "JigsawModelCustomHeader":
            header = config.model_kwargs.get("header", "maxpooling")
            num_freeze_layer = config.model_kwargs.get("num_freeze_layer", 12)
            model = JigsawModelCustomHeader(config.model_path, header=header, num_labels=2, num_freeze_layer=num_freeze_layer)
        elif config.model_class == "JigsawModelConcatrate":
            num_freeze_layer = config.model_kwargs.get("num_freeze_layer", 12)
            model = JigsawModelConcatrate(config.model_path, num_labels=2, num_freeze_layer=num_freeze_layer)
        else:
            model = AutoModelForSequenceClassification.from_pretrained(config.model_path, num_labels=2)

        # 訓練
        trainer = create_trainer(
            model=model,
            train_dataset=train_dataset,
            output_dir=config.output_dir,
            epochs=config.epochs,
            learning_rate=config.learning_rate,
            batch_size=config.batch_size,
            gradient_accumulation_steps=getattr(config, 'gradient_accumulation_steps', 6),
            use_irm=getattr(config, "use_irm", False),
            irm_lambda=getattr(config, "irm_lambda", 100.0),
            irm_n_envs=getattr(config, "irm_n_envs", 3),
        )

        trainer.train()

        # 予測保存
        submission_filename = f"submission_{config.name}.csv"
        predict_and_save(trainer, test_df, tokenizer, config.max_length, submission_filename)

        # メモリクリア
        del model, trainer, train_dataset
        torch.cuda.empty_cache()

        return submission_filename

    def train_single_qwen_model(self, config: ModelConfig) -> str:
        """Qwenモデルの訓練"""
        print(f"\n🚀 Training Qwen model: {config.name}")

        trainer = QwenModelTrainer(config)
        trainer.train()

        submission_filename = f"submission_{config.name}.csv"
        trainer.predict(submission_filename)

        return submission_filename

    def train_single_triplet_model(self, config: ModelConfig) -> str:
        """Tripletモデルの訓練"""
        print(f"\n🚀 Training Triplet model: {config.name}")

        trainer = TripletModelTrainer(config)
        trainer.train()

        submission_filename = f"submission_{config.name}.csv"
        trainer.predict(submission_filename)

        return submission_filename

    def train_all_models(self):
        """全モデルを順次訓練"""
        submission_files = []
        weights = []

        for config in self.model_configs:
            try:
                print(f"\n{'='*60}")
                print(f"Processing model: {config.name} (type: {config.model_type})")
                print(f"{'='*60}")

                if config.model_type == "deberta":
                    submission_file = self.train_single_deberta_model(config)
                elif config.model_type == "qwen":
                    submission_file = self.train_single_qwen_model(config)
                elif config.model_type == "triplet":
                    submission_file = self.train_single_triplet_model(config)
                else:
                    print(f"❌ Unknown model type: {config.model_type}")
                    continue

                submission_files.append(submission_file)
                weights.append(config.weight)

                print(f"✅ Model {config.name} completed successfully!")

            except Exception as e:
                print(f"❌ Error training {config.name}: {e}")
                import traceback
                traceback.print_exc()
                continue

        return submission_files, weights

def ensemble_predictions_advanced(submission_files: list, weights: list = None, output_file: str = "submission.csv"):
    """高度なアンサンブル予測"""
    if weights is None:
        weights = [1.0] * len(submission_files)

    # 重みを正規化
    weights = np.array(weights)
    weights = weights / weights.sum()

    def minmax_scale(series: pd.Series) -> pd.Series:
        """MinMax正規化"""
        s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
        fill = s.median()
        s = s.fillna(0.5 if pd.isna(fill) else fill)
        mn, mx = s.min(), s.max()
        if mx > mn:
            return (s - mn) / (mx - mn)
        return pd.Series(0.5, index=s.index, dtype=float)

    # データフレーム読み込みと結合
    ensemble_df = None

    for i, (file, weight) in enumerate(zip(submission_files, weights)):
        if not os.path.exists(file):
            print(f"⚠️ File not found: {file}")
            continue

        df = pd.read_csv(file)
        col_name = f"pred_{i}"
        df[col_name] = minmax_scale(df["rule_violation"])

        if ensemble_df is None:
            ensemble_df = df[["row_id", col_name]].copy()
        else:
            ensemble_df = ensemble_df.merge(df[["row_id", col_name]], on="row_id", how="outer")

        print(f"✓ Added {file} with weight {weight:.3f}")

    # アンサンブル計算
    pred_cols = [col for col in ensemble_df.columns if col.startswith("pred_")]
    ensemble_df[pred_cols] = ensemble_df[pred_cols].fillna(0.5)  # 欠損値を中央値で埋める

    ensemble_probs = np.zeros(len(ensemble_df))
    for i, col in enumerate(pred_cols):
        if i < len(weights):
            ensemble_probs += weights[i] * ensemble_df[col].values

    # 最終提出ファイル
    final_submission = pd.DataFrame({
        "row_id": ensemble_df["row_id"],
        "rule_violation": ensemble_probs
    })
    final_submission.to_csv(output_file, index=False)

    print(f"\n🎯 Final ensemble saved to {output_file}")
    print(f"Used files: {len(submission_files)}")
    print(f"Weights: {dict(zip(range(len(weights)), weights))}")

    return final_submission

def main():
    """メイン実行関数"""
    seed_everything(42)

    # モデル設定を取得
    all_configs = ModelRegistry.get_default_configs()
    enabled_configs = ModelRegistry.filter_enabled(all_configs)

    print(f"📊 Total models: {len(all_configs)}")
    print(f"📊 Enabled models: {len(enabled_configs)}")

    for config in enabled_configs:
        print(f"  - {config.name} ({config.model_type}) - weight: {config.weight}")

    # 統一トレーナーで実行
    trainer = UnifiedModelTrainer(enabled_configs)
    submission_files, weights = trainer.train_all_models()

    # アンサンブル
    if submission_files:
        ensemble_predictions_advanced(submission_files, weights)
        print(f"\n🎉 Training completed!")
        print(f"Individual submissions: {submission_files}")
        print(f"Final ensemble: submission.csv")
    else:
        print("❌ No successful model training!")

if __name__ == "__main__":
    main()

In [ ]:
cat submission.csv